<img src="https://userweb.fct.unl.pt/~jmc.xavier/MSII/iLogos/logo_novafct.png" width="200">

# Departamento de Engenharia Mecânica e Industrial
## Mecânica dos Sólidos II

## Análise e projeto de vigas sujeitas à flexão. Diagramas de esforços internos.

### Problema 5

A viga ACDEB indicada na figura é construída com um perfil laminado HEA300. Determine a intensidade do contrapeso P para o qual o valor absoluto máximo do momento fletor na viga é o menor possível e a correspondente tensão normal máxima devida à flexão.

<img src="https://userweb.fct.unl.pt/~jmc.xavier/MSII/Notebooks/Au04/P5/MSII_Au04_P5.png"
style="max-height: 100%; max-width: 100%;"/>

### Resolução


In [67]:
import numpy as np
import sympy as sy
from sympy.solvers import solve
import matplotlib.pyplot as plt
import os

cor = '2'
if cor == '1':
    plt.rcParams['axes.facecolor'] = (.15, .15, .15)
    plt.rcParams['figure.facecolor'] = (.15, .15, .15)
    plt.rcParams['font.family'] = 'monospace'
    plt.rcParams['font.size'] = 18
    # plt.rcParams['text.usetex'] = True
    params = {"ytick.color" : (.8, .8, .8),
              "xtick.color" : (.8, .8, .8),
              "grid.color" : (.2, .2, .2),
              "text.color" : (.7, .7, .7),
              "axes.labelcolor" : (.8, .8, .8),
              "axes.edgecolor" : (.15, .15, .15)}
else:
    plt.rcParams['axes.facecolor'] = (.7, .7, .7)
    plt.rcParams['figure.facecolor'] = (.7, .7, .7)
    plt.rcParams['font.family'] = 'monospace'
    plt.rcParams['font.size'] = 18
    # plt.rcParams['text.usetex'] = True
    params = {"ytick.color" : (.1, .1, .1),
              "xtick.color" : (.1, .1, .1),
              "grid.color" : (.2, .2, .2),
              "text.color" : (.1, .1, .1),
              "axes.labelcolor" : (.1, .1, .1),
              "axes.edgecolor" : (.15, .15, .15)}
plt.rcParams.update(params)

# data structure, units: N, mm, MPa

class varin: pass

dados = varin()
HEA300 = varin()

dados.L = 1. # unit: m
dados.Q = 10. # unit: kN

HEA300.Iz = 183.*1e6 # mm4
HEA300.bf = 290. # mm - largura do banzo
HEA300.Wz = (1260.*1e3)*1e-9 # m³ - largura do banzo

- Equilíbrio estático

In [48]:
ra, rb, p = sy.symbols('ra rb p')

sumFy = ra + rb + -2*dados.Q + p
print('sumFy = ',sumFy)
sumMD = - ra*(2*dados.L) + rb*(2*dados.L) + dados.Q*dados.L - dados.Q*dados.L
print('sumMD = ',sumMD)
sumMC = - ra*(dados.L) + rb*(3*dados.L) + p*(dados.L) - dados.Q*(2*dados.L)
print('sumMC = ',sumMC)
sol = solve({sumFy,sumMD,sumMC},{ra, rb, p})

P = sol[p]
print('sol. ----------------------')
print(f'P  = {P} [kN]')
print(f'ra = {sol[ra]} [kN]')

sumFy =  p + ra + rb - 20
sumMD =  -2*ra + 2*rb
sumMC =  p - ra + 3*rb - 20
sol. ----------------------
P  = 20 - 2*rb [kN]
ra = rb [kN]


\begin{equation}
R_A = R_B = R
\quad\wedge\quad
P = 20-2R
\end{equation}

In [38]:
r = sy.symbols('r')

R = solve(sumFy.subs({(ra,r),(rb,r)}),{r})[0]
print(f'R = {R}')

R = 10 - p/2


- Diagrama de esforços: momento fletor

- Tramo AC

    - Momento fletor:

\begin{equation}
\begin{split}
&- R_A \cdot x + M_{AC}(x) = 0 \\[1ex]
\therefore&\quad
M_C \equiv M_{AC}(x_C) = R_A L_{AC}
\end{split}
\end{equation}

In [51]:
x = sy.symbols('x')

mac = R*x
print(f'MAC(x) = {mac} [kN.m]')
mac_c = mac.subs({(x,dados.L)})
print(f'MC = {mac_c} [kN.m]')

MAC(x) = x*(10 - p/2) [kN.m]
MC = 10 - p/2 [kN.m]


- Tramo CD

    - Momento fletor:

\begin{equation}
\begin{split}
&- R_A ~x + Q(x-L_{AC}) + M_{CD} = 0 \\[1ex]
\therefore&\quad M_{CD} = (R_A - Q) x + QL_{AC}\\[1ex]
\therefore&\quad
M_D \equiv M_{CD}(x_D) = (R_A-Q)L_{AD} + QL_{AC}
\end{split}
\end{equation}

In [59]:
mcd = -(R*x - dados.Q*(x-dados.L))
# negativo por ser avaliado em módulo (!)
print(f'MCD(x) = {sy.factor(mcd)} [kN.m]')
mcd_d = mcd.subs({(x,2*dados.L)})
print(f'MD = {mcd_d} [kN.m]')

MCD(x) = (p*x - 20)/2 [kN.m]
MD = p - 10 [kN.m]


- A condição da intensidade do contrapeso $P$ para o qual o
valor absoluto máximo do momento fletor na viga é o
menor possível é dada por:

\begin{equation}
|M_\mathrm{max}| = |M_\mathrm{min}|
\Leftrightarrow
M_\mathrm{C} = M_\mathrm{D}
\end{equation}

In [65]:
EQ = mac_c - mcd_d
sol = solve(EQ,p)
Psol = sol[0]
print(f'P = {Psol} = {float(Psol):.1f} kN')

P = 40/3 = 13.3 kN


In [86]:
Mmax = mcd_d.subs({(p,Psol)})*1e3
print(f'M.max = {float(Mmax):.1f} [N.m]')
print(f'Wz(HEA300) = {HEA300.Wz:.3e} [m³]')
sig = Mmax/HEA300.Wz
print(f'Sig.max = {sig*1e-6:.3f} [MPa]')

M.max = 3333.3 [N.m]
Wz(HEA300) = 1.260e-03 [m³]
Sig.max = 2.646 [MPa]


---

Copyright (c) DEMI - FCT NOVA

Interactive computing by <a href="https://jupyter.org/" target="_blank"> <span
style="color:#333399"> Jupyter Notebook </span> </a> &nbsp;|&nbsp;Coded by <a href = "mailto: jmc.xavier@fct.unl.pt">José Xavier</a>

Licensed under  <a href="http://creativecommons.org/licenses/by-sa/4.0/"
target="_blank"> <span style="color:#333399;font-size: 20px"> CC BY-SA 4.0  </span></a>